# Test path classification model

Initialize the device to use for model inference

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("is cuda available:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Initialize data paths and split that were used during training

In [ ]:
import os
from image_segmentation.data import ImageDataset
from utils.available_datasets import available_datasets

dataset_choice = available_datasets["FIVES"]

dataset_name = dataset_choice.name
train_split = dataset_choice.preferred_train_split
data_dir = dataset_choice.data_dir

dataset = ImageDataset(data_dir=data_dir)
distances_hparams = dataset.get_dataset_distance_hparams()

Initialize the model components with the same hyperparameters as during training

In [ ]:
from path_neural_networks.models.features_generators import FeaturesGenerator, PretrainedUnetFeaturesGenerator
from path_neural_networks.models.path_samplers import PathSampler, MultiScaleSquarePathSampling, SamplingMaxAggregation
from path_neural_networks.models.path_encoders import PathEncoder, ConvMaxPoolingPathEncoder
from path_neural_networks.models.path_classifiers import PathClassifier, FCNPathClassifier
from utils.other import pretty_dict_print

used_provided_unet = True

unet_checkpoint_dir = dataset_choice.value.get_checkpoint_dir("unet_pretrained" if used_provided_unet else "unet_pretraining")
unet_ckpt_path = os.path.join(unet_checkpoint_dir, os.listdir(unet_checkpoint_dir)[0])
print("Using UNet checkpoint:", unet_ckpt_path)

features_generator_out_channels = 32
sampling_square_sizes = distances_hparams["scales_sampling_sizes"]
sampling_aggregation_method = SamplingMaxAggregation()
conv_path_residual_blocks = False
conv_path_skip_connections = False
conv_path_layers = [None, None, 256]
path_classifier_n_hidden_layers = 2
path_classifier_dropout = 0

features_generator: FeaturesGenerator = PretrainedUnetFeaturesGenerator(
    ckpt_path=unet_ckpt_path,
    device=device,
    out_channels=features_generator_out_channels,
    freeze_pretrained=False,
    skip_connection=False
)
path_sampler: PathSampler = MultiScaleSquarePathSampling(
    in_channels=features_generator_out_channels,
    square_sizes=sampling_square_sizes,
    aggregation=sampling_aggregation_method
)
path_encoder: PathEncoder = ConvMaxPoolingPathEncoder(
    in_channels=path_sampler.out_channels, 
    hidden_layers=conv_path_layers, 
    skip_connection=conv_path_skip_connections, 
    residual_blocks=conv_path_residual_blocks
)
path_classifier: PathClassifier = FCNPathClassifier(
    in_channels=path_encoder.out_channels,
    n_hidden_layers=path_classifier_n_hidden_layers,
    num_classes=1,
    dropout=path_classifier_dropout
)

Initialize datamodules

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
from math import ceil

from path_neural_networks.data.image_centerline_dataset import ImageCenterlineDataset
from path_neural_networks.data.image_centerline_datamodule import ImageCenterlineDatamodule
from path_neural_networks.data.augmentations import build_val_transform

max_dist = int(ceil(distances_hparams['max_dist']))
if max_dist is None:
    centerlines_dirname = "euclidean_all_centerlines"
else:
    centerlines_dirname = f"euclidean_lt_{max_dist}_centerlines"

split_file_path=os.path.join(data_dir, "splits.json")
val_split_ratio = 0.2
use_foreground_pixels_only_for_normalization = True
data_seed = 42
split_seed = 42

dataset = ImageCenterlineDataset(data_dir=data_dir, centerline_dirname=centerlines_dirname)
datamodule = ImageCenterlineDatamodule(dataset=dataset, 
                                        split_file_path=split_file_path,
                                        train_split_name=train_split,
                                        val_split_ratio=val_split_ratio,
                                        seed = split_seed)
datamodule.setup()

stats = dataset.get_dataset_stats(split_name=train_split, split_indices=datamodule.train_indices.tolist() + datamodule.val_indices.tolist())
if use_foreground_pixels_only_for_normalization:
    stats = stats['foreground']
else:
    stats = stats['full_image']
mean, std = stats['mean'], stats['std']
print("Dataset stats used for normalization:")
pretty_dict_print(stats)

test_transforms = build_val_transform(mean, std)

datamodule = ImageCenterlineDatamodule(dataset=dataset,
                                        split_file_path=split_file_path,
                                        train_split_name=train_split,
                                        val_split_ratio=val_split_ratio,
                                        val_transforms=test_transforms,
                                        test_transforms=test_transforms,
                                        seed = split_seed)

Initialize the loss function

In [ ]:
from path_neural_networks.models.losses import PathClassificationLoss, WeightedBCEWithLogitsLoss, BCEWithLogitsLoss

use_pos_weight_in_loss = True

loss_fn: PathClassificationLoss
if use_pos_weight_in_loss:
    classes_stats = dataset.get_dataset_classes_stats()
    classes_ratio = classes_stats['classes_ratio']
    loss_fn = WeightedBCEWithLogitsLoss(classes_ratio=classes_ratio)
else:
    loss_fn = BCEWithLogitsLoss()

## Initialize the model checkpoint, two choices:

### Case #1: Use the provided weights (see [README](../README.md)) to initialize the pretrained path classification model

In [ ]:
checkpoint_dir = os.path.abspath(os.path.join("..", "checkpoints", dataset_name, "main_model_pretrained"))
ckpt_path = os.path.join(checkpoint_dir, os.listdir(checkpoint_dir)[0])
print("Using UNet checkpoint:", ckpt_path)

### Case #2: Use the model trained in the [preceding notebook (5)](./05_train_path_classification_model.ipynb)

In [ ]:
checkpoint_dir = os.path.abspath(os.path.join("..", "checkpoints", dataset_name, "main_model_training"))
ckpt_path = os.path.join(checkpoint_dir, os.listdir(checkpoint_dir)[0])
print("Using UNet checkpoint:", ckpt_path)

## Load model and trainer

In [ ]:
from path_neural_networks.models import ReducedPipelineLitModule
from path_neural_networks.utils.symmetry_enforcement import SymmetryEnforcementMode

metrics = ["accuracy", "auroc", "recall", "precision", "pr_auc"]
symmetry_enforcement_mode = SymmetryEnforcementMode.NONE

model = ReducedPipelineLitModule.load_from_checkpoint(
    ckpt_path,
    features_generator=features_generator,
    path_sampler=path_sampler,
    path_encoder=path_encoder,
    path_classifier=path_classifier,
    edge_classification_loss_fn=loss_fn,
    metrics=metrics,
    symmetry_enforcement_mode=symmetry_enforcement_mode
)

In [ ]:
from pytorch_lightning import Trainer

torch.set_float32_matmul_precision("medium")
trainer = Trainer(accelerator='gpu', devices="auto", max_epochs=10, precision='16-mixed')

## Testing the trained model

In [ ]:
trainer.test(model, datamodule=datamodule)

### Finding the best threshold on the validation set

In [ ]:
from tqdm import tqdm

def perform_inference(model, dataloader, apply_sigmoid: bool = True):
    all_preds = []
    all_targets = []

    model.eval()
    with torch.no_grad():
        for batch in tqdm(dataloader):
            img, (paths, edges_classes), _ = batch
            path_logits, _ = model(img, paths)
            all_preds.append(path_logits.detach().cpu())
            assert path_logits.shape[0] == edges_classes.shape[1], f"Number of edge predictions ({path_logits.shape[0]}) does not match number of edge labels ({edges_classes.shape[1]})"
            all_targets.append(edges_classes.squeeze().detach().cpu())

    all_preds = torch.cat(all_preds)
    all_targets = torch.cat(all_targets)
    if apply_sigmoid:
        all_preds = torch.sigmoid(all_preds)

    print(f"Performed inference on {len(dataloader)} cases, and on a total of {all_preds.shape[0]} paths.")
    return all_preds, all_targets

In [ ]:
preds, targets = perform_inference(model, datamodule.val_dataloader())

We compute the threshold that maximizes the F1 score on the validation set, to use it later for inference on the test set

In [ ]:
from torchmetrics.classification import BinaryPrecisionRecallCurve

pr_curve = BinaryPrecisionRecallCurve()
precision, recall, thresholds = pr_curve(preds, targets)

max_f1_threshold = thresholds[torch.argmax(2 * (precision * recall) / (precision + recall + 1e-8))]

print(max_f1_threshold)

## Quantitative analysis

In [ ]:
model.set_inference_threshold(max_f1_threshold.item())
preds, targets = perform_inference(model, datamodule.test_dataloader())

As it was done in the article, we compare our trained model to a naive model, based on the distance between the endpoints to connect, without any image-based features.

### Naive model initialization and inference

We first want to find the optimal distance threshold for this naive model

In [ ]:
from path_neural_networks.models.naive_model import NaiveModelLitModule
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import numpy as np
import matplotlib.pyplot as plt

naive_accuracy_scores = []
naive_precision_scores = []
naive_recall_scores = []
naive_f1_scores = []

x_range = np.arange(0, 105, 5)

for dist_threshold in x_range:
    print(f"Evaluating Naive Model with distance threshold: {dist_threshold}")
    naive_model = NaiveModelLitModule(max_dist=dist_threshold, metrics=metrics)

    naive_preds, naive_targets = perform_inference(naive_model, datamodule.val_dataloader(), apply_sigmoid=False)

    naive_accuracy_scores.append(accuracy_score(naive_targets.numpy(), naive_preds.numpy()))
    naive_precision_scores.append(precision_score(naive_targets.numpy(), naive_preds.numpy(), zero_division=1.0))
    naive_recall_scores.append(recall_score(naive_targets.numpy(), naive_preds.numpy()))
    naive_f1_scores.append(f1_score(naive_targets.numpy(), naive_preds.numpy()))

naive_max_f1_threshold_index = np.argmax(naive_f1_scores)
naive_max_f1_threshold = x_range[naive_max_f1_threshold_index]

plt.plot(x_range, naive_accuracy_scores, label="Accuracy")
plt.plot(x_range, naive_precision_scores, label="Precision")
plt.plot(x_range, naive_recall_scores, label="Recall")
plt.plot(x_range, naive_f1_scores, label="F1 Score")
plt.axvline(x=naive_max_f1_threshold, color='r', linestyle='--', label=f'Max F1 - Distance Threshold: {naive_max_f1_threshold}')
plt.xlabel("Distance Threshold")
plt.ylabel("Score")
plt.title("Naive Model Performance vs Distance Threshold")
plt.legend()
plt.grid()
plt.show()

naive_max_f1_precision = naive_precision_scores[naive_max_f1_threshold_index]
naive_max_f1_recall = naive_recall_scores[naive_max_f1_threshold_index]
naive_max_f1_accuracy = naive_accuracy_scores[naive_max_f1_threshold_index]
naive_max_f1 = naive_f1_scores[naive_max_f1_threshold_index]

print(f"Naive Model - Max F1 Score: {naive_max_f1:.4f} at Distance Threshold: {naive_max_f1_threshold}, with Precision: {naive_max_f1_precision:.4f}, Recall: {naive_max_f1_recall:.4f}, Accuracy: {naive_max_f1_accuracy:.4f}")

In [ ]:
from torchmetrics.classification import BinaryPrecisionRecallCurve, BinaryROC
from torchmetrics.utilities.compute import auc
import matplotlib.pyplot as plt

# ROC Curve
roc_curve = BinaryROC()
fpr, tpr, roc_thresholds = roc_curve(preds, targets)
auroc = auc(fpr, tpr)

# Precision-Recall Curve
pr_curve = BinaryPrecisionRecallCurve()
precision, recall, thresholds = pr_curve(preds, targets)
pr_auc = auc(recall, precision)

naive_recall_scores = torch.tensor(naive_recall_scores)
naive_precision_scores = torch.tensor(naive_precision_scores)
naive_pr_auc = auc(naive_recall_scores, naive_precision_scores)


# find the tpr and fpr at the best F1 threshold
best_tpr = tpr[torch.argmin(torch.abs(roc_thresholds - max_f1_threshold))].item()
best_fpr = fpr[torch.argmin(torch.abs(roc_thresholds - max_f1_threshold))].item()

# find the precision and recall at the best F1 threshold
best_precision = precision[torch.argmax(2 * (precision * recall) / (precision + recall + 1e-8))].item()
best_recall = recall[torch.argmax(2 * (precision * recall) / (precision + recall + 1e-8))].item()

# ============= Visualization =============

fig, axs = plt.subplots(1, 2, figsize=(14, 4))

axs[0].plot(fpr.numpy(), tpr.numpy(), label=f"Our model (AUROC={auroc.item():.4f})")
axs[0].plot([0, 1], [0, 1], 'k--', label="Random classifier")
axs[0].fill_between([0, 1], [0, 1], alpha=0.1, color='blue')
axs[0].fill_between(fpr.numpy(), tpr.numpy(), alpha=0.1, color='blue')
axs[0].scatter(best_fpr, best_tpr, color='m', label=f"Our model - Best F1 point ($\\tau$={max_f1_threshold.item():.4f})")
axs[0].set_xlabel("False Positive Rate")
axs[0].set_ylabel("True Positive Rate")
axs[0].set_title("ROC Curve (Test set)")
axs[0].grid(True)
axs[0].legend(loc="center right", bbox_to_anchor=(0.95, 0.2))

axs[1].plot(recall.numpy(), precision.numpy(), label=f"Our model (AUC={pr_auc.item():.4f})")
axs[1].fill_between(recall.numpy(), precision.numpy(), alpha=0.1, color='blue')
axs[1].plot(naive_recall_scores.numpy(), naive_precision_scores.numpy(), label=f"Naive model (AUC={naive_pr_auc.item():.4f})")
axs[1].fill_between(naive_recall_scores.numpy(), naive_precision_scores.numpy(), alpha=0.1, color='blue')
axs[1].scatter(best_recall, best_precision, color='m', label=f"Our model - Best F1 point ($\\tau$={max_f1_threshold.item():.4f})")
axs[1].scatter(naive_max_f1_recall, naive_max_f1_precision, color='red', label=f"Naive model - Best F1 point (d = {naive_max_f1_threshold})")
axs[1].set_xlabel("Recall")
axs[1].set_ylabel("Precision")
axs[1].set_title("Precision-Recall Curve (Test set)")
axs[1].grid(True)
axs[1].legend(loc="center right", bbox_to_anchor=(0.72, 0.22))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

eps = 1e-8
thresholds_dense = torch.quantile(preds, torch.linspace(0, 1, steps=100))
thresholds_dense = torch.unique(thresholds_dense)

targets = targets.bool()
dice_scores = []
tp_list, fp_list, tn_list, fn_list = [], [], [], []
for t in thresholds_dense:
    preds_b = (preds >= t)

    tp = (preds_b & targets).sum().float()
    fp = (preds_b & ~targets).sum().float()
    tn = (~preds_b & ~targets).sum().float()
    fn = (~preds_b & targets).sum().float()

    dice = (2 * tp) / (2 * tp + fp + fn + eps)

    dice_scores.append(dice)
    tp_list.append(tp)
    fp_list.append(fp)
    tn_list.append(tn)
    fn_list.append(fn)

dice_scores = torch.stack(dice_scores)
tp, fp, tn, fn = torch.stack(tp_list), torch.stack(fp_list), torch.stack(tn_list), torch.stack(fn_list)
true_class_0_total = tn + fp
true_class_1_total = tp + fn
tp_prop = tp / true_class_1_total
fp_prop = fp / true_class_0_total
tn_prop = tn / true_class_0_total
fn_prop = fn / true_class_1_total

# Confusion Matrix at the best F1 threshold
cm = confusion_matrix(targets.numpy(), (preds.numpy() > max_f1_threshold.item()).astype(int))
cm = cm / cm.sum(axis=1, keepdims=True)  # Normalize by true class counts
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])

fig, axs = plt.subplots(1, 3, figsize=(20, 5))

axs[0].plot(thresholds_dense.numpy(), fp_prop.numpy()) 
axs[0].fill_between(thresholds_dense.numpy(), fp_prop.numpy(), y2=0.0, alpha=0.1, color='red', label="FP area")
axs[0].fill_between(thresholds_dense.numpy(), fp_prop.numpy(), y2=1.0, alpha=0.1, color='green', label="TN area")
axs[0].axvline(x=max_f1_threshold.item(), color='red', linestyle='--', label=f"Best F1 threshold: {max_f1_threshold.item():.4f}") 
axs[0].set_xlabel("Threshold") 
axs[0].set_ylabel("Proportion of total samples for true class 0") 
axs[0].set_title("Confusion Matrix (True class 0) Proportions vs Threshold") 
axs[0].legend()
axs[0].grid(True)

axs[1].plot(thresholds_dense.numpy(), fn_prop.numpy()) 
axs[1].fill_between(thresholds_dense.numpy(), fn_prop.numpy(), y2=0.0, alpha=0.1, color='red', label="FN area")
axs[1].fill_between(thresholds_dense.numpy(), fn_prop.numpy(), y2=1.0, alpha=0.1, color='green', label="TP area")
axs[1].axvline(x=max_f1_threshold.item(), color='red', linestyle='--', label=f"Best F1 threshold: {max_f1_threshold.item():.4f}") 
axs[1].set_xlabel("Threshold") 
axs[1].set_ylabel("Proportion of total samples for true class 1") 
axs[1].set_title("Confusion Matrix (True class 1) Proportions vs Threshold") 
axs[1].legend()
axs[1].grid(True)

disp.plot(ax=axs[2], values_format='.2f', cmap='Blues')
axs[2].set_title(f"Confusion Matrix (Test set, threshold={max_f1_threshold.item():.4f})")

plt.show()

## Qualitative analysis

In [ ]:
import networkx as nx
import torch

from graph.graph_creation import img_to_graph
from graph.graph_oversampling import OversampleNodesTransform
from graph.graph_wrapper import GraphWrapper
from path_neural_networks.utils.paths_creation import cut_mask_from_negative_edges_for_all, get_query_edges
from utils.reconstruction.path_reconstruction.euclidean_path_reconstruction import EuclideanPathReconstructionMethod

path_reconstruction_method = EuclideanPathReconstructionMethod()

def process_case(pred: torch.Tensor, 
                 centerline_max_dist: float,
                 oversampling_max_dist: float,
                 n_closest: int,
                 display: bool = False) -> None:
    pred_np = pred.squeeze().cpu().numpy() > 0
    
    pred_graph: nx.Graph = img_to_graph(pred_np, clean = True, closing_radius=1, return_pixel_graph=False)
    oversample_nodes_transform = OversampleNodesTransform(oversampling_max_dist, remove_original_edges=True)
    graph_wrapper: GraphWrapper = oversample_nodes_transform(GraphWrapper(pred_graph))
    new_nx_graph: nx.Graph = graph_wrapper.get_graph()

    nodes_radius_data = {}
    for id, n_data in new_nx_graph.nodes(data=True):
        nodes_radius_data[id] = float(n_data["radius"])

    virtual_edges_index_tensor = get_query_edges(new_nx_graph, n_closest=n_closest, max_dist=centerline_max_dist)
    eval_edges = virtual_edges_index_tensor.t().tolist()
    eval_path_centerlines = path_reconstruction_method.reconstruct(map=None, graph=new_nx_graph, new_edges=virtual_edges_index_tensor)
    eval_path_centerlines = [[list([int(x), int(y)]) for (x, y) in path] for path in eval_path_centerlines]
    res = cut_mask_from_negative_edges_for_all(eval_path_centerlines, pred_np, edges=eval_edges, return_old_centerlines=True)
    eval_path_centerlines_cut = res["new_centerlines"]
    eval_path_centerlines = res["old_centerlines"]
    eval_edges = res["edges"]

    eval_data = {
        "edges": eval_edges,
        "nodes_radius": nodes_radius_data,
        "path_centerlines": eval_path_centerlines_cut,
        "full_path_centerlines": eval_path_centerlines
    }

    return eval_data

In [ ]:
oversampling_max_dist = 50.0
n_closest = 5

nb_cases_to_display = 5
model.eval()
with torch.no_grad():
    for i, batch in enumerate(tqdm(datamodule.test_dataloader())):
        if i >= nb_cases_to_display:
            break
        img, _, (_, pred) = batch
        eval_data = process_case(pred, centerline_max_dist=max_dist, oversampling_max_dist=oversampling_max_dist, n_closest=n_closest)
        paths = eval_data["path_centerlines"]
        paths = [torch.tensor(path, dtype=torch.float32).unsqueeze(0) for path in paths]
        path_logits, feature_maps = model(img, paths)

        path_probs = torch.sigmoid(path_logits)
        path_preds = (path_probs > max_f1_threshold).squeeze().cpu().numpy()
        print(f"Number of paths: {len(paths)}, Number of predicted positive paths: {path_preds.sum()}")

        img_np = img.squeeze().cpu().numpy().transpose(1, 2, 0)
        img_np = (img_np * std) + mean  # Denormalize for visualization

        path_img = np.zeros_like(img_np)
        path_img[pred.squeeze().bool()] = [1.0, 1.0, 1.0]
        for path, pred in zip(paths, path_preds):
            path = path.squeeze(0).cpu().numpy().astype(int)
            print(path.shape)
            if pred:
                path_img[path[:, 0], path[:, 1]] = [0.0, 1.0, 0.0]  # Green for positive paths
            else:
                path_img[path[:, 0], path[:, 1]] = [1.0, 0.0, 0.0]  # Red for negative paths

        fig, axs = plt.subplots(1, 2, figsize=(20, 20))
        axs[0].imshow(img_np)
        axs[1].imshow(path_img)
        plt.show()